# 04_Modeling

Playground Series S6E7 — Predicting Student Health Risk

Goal: train and compare a handful of baseline models using cross-validation,
pick the best one, and export a **checkpoint submission** — just to confirm
the CV score roughly matches the public leaderboard score (a sanity check,
not the final tuned submission).

Input: `processed_train.csv`, `processed_test.csv` from `03_Preprocessing.ipynb`
(already imputed, encoded, and feature-selected)

Models compared: RandomForest, CatBoost, LightGBM (XGBoost optional — see note below)

The final, fully tuned submission (after `05_Feature_Engineering.ipynb`) belongs
in `06_Final_Submission.ipynb`, not here.

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import balanced_accuracy_score

pd.set_option('display.max_columns', None)
RANDOM_STATE = 42
N_SPLITS = 5

## 1. Load data

In [2]:
processed_train = pd.read_csv('../data/processed/processed_train.csv')
processed_test = pd.read_csv('../data/processed/processed_test.csv')

y = processed_train['health_condition']
X = processed_train.drop(columns=['health_condition'])
X_test = processed_test.copy()

print('X shape:', X.shape)
print('X_test shape:', X_test.shape)
print('Class distribution:')
print(y.value_counts(normalize=True).round(3))

X shape: (690088, 16)
X_test shape: (295753, 16)
Class distribution:
health_condition
1    0.859
2    0.084
0    0.058
Name: proportion, dtype: float64


## 2. Define models to compare
Each entry is a factory function so a fresh, unfitted model is created for every fold
(reusing the same fitted model across folds would leak information between folds).

In [ ]:
def make_random_forest():
    return RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight='balanced'
    )

def make_catboost():
    return CatBoostClassifier(
        iterations=1500,
        learning_rate=0.05,
        depth=8,
        random_state=RANDOM_STATE,
        verbose=False,
        early_stopping_rounds=100,
        auto_class_weights='Balanced'
    )

def make_lightgbm():
    return LGBMClassifier(
        n_estimators=1500,
        learning_rate=0.05,
        max_depth=8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight='balanced'
    )

# XGBoost is optional — uncomment if you have it installed and want a 4th candidate
# from xgboost import XGBClassifier
# def make_xgboost():
#     return XGBClassifier(
#         n_estimators=1500, learning_rate=0.05, max_depth=8,
#         random_state=RANDOM_STATE, n_jobs=-1, eval_metric='mlogloss'
#     )

model_factories = {
    'RandomForest': make_random_forest,
    'CatBoost': make_catboost,
    'LightGBM': make_lightgbm,
    # 'XGBoost': make_xgboost,
}

## 3. Cross-validation comparison
StratifiedKFold keeps the class proportions consistent across folds — important here
since the target has 3 classes that may not be perfectly balanced.

CatBoost and LightGBM use `eval_set` + early stopping so `n_estimators`/`iterations`
above are just an upper bound, not a fixed number of rounds.

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

cv_results = {name: [] for name in model_factories}
oof_preds = {name: np.zeros(len(X)) for name in model_factories}

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    print(f'\n=== Fold {fold + 1}/{N_SPLITS} ===')

    for name, factory in model_factories.items():
        model = factory()

        if name in ('CatBoost',):
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
        elif name in ('LightGBM',):
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)])
        else:
            model.fit(X_tr, y_tr)

        val_pred = model.predict(X_val)
        val_pred = np.asarray(val_pred).flatten()
        acc = balanced_accuracy_score(y_val, val_pred)

        cv_results[name].append(acc)
        oof_preds[name][val_idx] = val_pred

        print(f'  {name:<15} fold balanced_acc = {acc:.5f}')


=== Fold 1/5 ===
  RandomForest    fold acc = 0.96460
  CatBoost        fold acc = 0.96503


c:\Users\thach\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009355 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1805
[LightGBM] [Info] Number of data points in the train set: 552070, number of used features: 16
[LightGBM] [Info] Start training from score -2.852889
[LightGBM] [Info] Start training from score -0.152364
[LightGBM] [Info] Start training from score -2.481150
  LightGBM        fold acc = 0.96508

=== Fold 2/5 ===
  RandomForest    fold acc = 0.96514
  CatBoost        fold acc = 0.96581


c:\Users\thach\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005931 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1806
[LightGBM] [Info] Number of data points in the train set: 552070, number of used features: 16
[LightGBM] [Info] Start training from score -2.852889
[LightGBM] [Info] Start training from score -0.152364
[LightGBM] [Info] Start training from score -2.481150
  LightGBM        fold acc = 0.96585

=== Fold 3/5 ===
  RandomForest    fold acc = 0.96495
  CatBoost        fold acc = 0.96543


...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007124 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1805
[LightGBM] [Info] Number of data points in the train set: 552070, number of used features: 16
[LightGBM] [Info] Start training from score -2.852889
[LightGBM] [Info] Start training from score -0.152364
[LightGBM] [Info] Start training from score -2.481150
  LightGBM        fold acc = 0.96541

=== Fold 4/5 ===
  RandomForest    fold acc = 0.96460
  CatBoost        fold acc = 0.96513


c:\Users\thach\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007335 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1805
[LightGBM] [Info] Number of data points in the train set: 552071, number of used features: 16
[LightGBM] [Info] Start training from score -2.852859
[LightGBM] [Info] Start training from score -0.152366
[LightGBM] [Info] Start training from score -2.481152
  LightGBM        fold acc = 0.96508

=== Fold 5/5 ===
  RandomForest    fold acc = 0.96435
  CatBoost        fold acc = 0.96499


c:\Users\thach\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.029894 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1805
[LightGBM] [Info] Number of data points in the train set: 552071, number of used features: 16
[LightGBM] [Info] Start training from score -2.852859
[LightGBM] [Info] Start training from score -0.152368
[LightGBM] [Info] Start training from score -2.481130
  LightGBM        fold acc = 0.96501


## 4. Compare CV scores across models (balanced accuracy)

In [ ]:
summary = pd.DataFrame({
    name: {
        'mean_bal_acc': np.mean(scores),
        'std_bal_acc': np.std(scores),
        'oof_bal_acc': balanced_accuracy_score(y, oof_preds[name])
    }
    for name, scores in cv_results.items()
}).T.sort_values('oof_bal_acc', ascending=False)
summary

,mean_bal_acc,std_bal_acc,oof_bal_acc
LightGBM,0.965287,0.000315,0.857974
CatBoost,0.965280,0.000306,0.856134
RandomForest,0.964728,0.000281,0.847558


In [ ]:
best_model_name = summary.index[0]
print(f'Best model by OOF balanced accuracy: {best_model_name}')
print(classification_report(y, oof_preds[best_model_name]))

Best model by OOF accuracy: LightGBM
              precision    recall  f1-score   support

           0       0.95      0.80      0.87     39803
           1       0.97      0.99      0.98    592561
           2       0.96      0.78      0.86     57724

    accuracy                           0.97    690088
   macro avg       0.96      0.86      0.90    690088
weighted avg       0.97      0.97      0.96    690088



## 5. Checkpoint submission
Refit the best model on the FULL training set (not just one fold), predict on test,
and export a submission — purely to sanity-check that the public leaderboard score
is in the same ballpark as the OOF balanced accuracy above. If it's way off, something
in the pipeline (leakage, train/test mismatch, or a metric misunderstanding) needs
investigating before moving on.

In [ ]:
final_model = model_factories[best_model_name]()

if best_model_name == 'CatBoost':
    final_model.fit(X, y, verbose=False)
elif best_model_name == 'LightGBM':
    final_model.fit(X, y)
else:
    final_model.fit(X, y)

test_pred = np.asarray(final_model.predict(X_test)).flatten()

# reverse the target mapping from 03_Preprocessing (fit:0, at-risk:1, unhealthy:2)
inverse_mapping = {0: 'fit', 1: 'at-risk', 2: 'unhealthy'}
test_pred_labels = pd.Series(test_pred).map(inverse_mapping)

sample_submission = pd.read_csv('../data/sample_submission.csv')
checkpoint_submission = sample_submission.copy()
checkpoint_submission['health_condition'] = test_pred_labels.values

checkpoint_submission.to_csv(
    f'../submission/checkpoint_{best_model_name.lower()}.csv',
    index=False
)
checkpoint_submission.head()

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006912 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1805
[LightGBM] [Info] Number of data points in the train set: 690088, number of used features: 16
[LightGBM] [Info] Start training from score -2.852877
[LightGBM] [Info] Start training from score -0.152365
[LightGBM] [Info] Start training from score -2.481146


,id,health_condition
0,690088,unhealthy
1,690089,at-risk
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


## Summary

- Compared RandomForest, CatBoost, LightGBM with 5-fold Stratified CV, scored on
  **balanced accuracy** (the actual competition metric) instead of plain accuracy
- All 3 models use class-balancing so minority classes (fit, unhealthy) aren't ignored
- Best model: see `summary` table above (by OOF balanced accuracy)
- Exported a checkpoint submission — compare its Kaggle public score against the
  OOF balanced accuracy above; they should now be much closer than before
  (previously: OOF plain accuracy 0.965 vs. public score 0.855 — a metric mismatch,
  not a bug)

Next: `05_Feature_Engineering.ipynb`, using this notebook's CV setup + feature
importance to decide which new features are worth creating, then
`06_Final_Submission.ipynb` for the actual tuned, official submission.